In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
from torch.optim import Adam
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)

print(f"X_train shape {X_train.shape}")
print(f"X_test shape {X_test.shape}")
print(f"y_train shape {y_train.shape}")
print(f"y_test shape {y_test.shape}")


In [ ]:
# 2. Create TensorDataset objects

train_data = TensorDataset(X_train, y_train)
test_data = TensorDataset(X_test, y_test)


In [ ]:
# 3. Create DataLoaders

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64,)


In [ ]:
# 4. Print shape of one batch

for X_batch, y_batch in train_loader:

  print(f"X_batch Shape {X_batch.shape}")
  print(f"y_batch Shape{y_batch.shape}")

  break

for X_batch_test, y_batch_test in test_loader:

  print(f"X_batch Shape {X_batch_test.shape}")
  print(f"y_batch Shape{y_batch_test.shape}")

  break

In [ ]:
# 5. Display sample images


In [ ]:
# Task 1: Write your model class here:

class NN4Layer(nn.Module):

  def __init__(self, input_dim, hidden_dim, output_dim):
    super().__init__()
    self.layer1 = nn.Linear(input_dim, hidden_dim)
    self.layer2 = nn.Linear(hidden_dim, hidden_dim)
    self.layer3 = nn.Linear(hidden_dim, hidden_dim)
    self.layer4 = nn.Linear(hidden_dim, output_dim)
  def forward(self, x):
    a1 = self.layer1(x)
    a2 = self.layer2(a1)
    a3 = self.layer3(a2)
    a4 = self.layer3(a3)

    return a4



In [ ]:
# Task 2: Write your training loop here:

def train_one_epoch(model, optimizer, loss_fn, train_loader, device):

  model.train()

  train_loss = 0

  for batch_X, batch_y in train_loader:

    batch_X = batch_X.view(X_batch.size(0), -1).to(device)
    batch_y = batch_y.to(device)

    y_pred = model(batch_X)

    loss = loss_fn(y_pred, batch_y)
    model.zero_grad()
    loss.backward()
    optimizer.step()

    train_loss += loss.item()

  avg_train_loss = train_loss/len(train_loader)

  return avg_train_loss

  # avg_train_losses.append(avg_train_loss)



In [ ]:
# Task 3: Write your validation loop here:

def validate(model, loss_fn, test_loader, device):

  model.eval()

  with torch.inference_mode():

    val_loss = 0
    correct = 0

    for val_batch_X, val_batch_y in test_loader:

      val_batch_X = val_batch_X.view(X_batch.size(0), -1).to(device)
      val_batch_y = val_batch_y.to(device)

      val_y_pred = model(val_batch_X)

      loss = loss_fn(val_y_pred, val_batch_y)
      val_loss += loss.item()

      correct += (val_y_pred == val_batch_y).sum().item()

  avg_val_loss = val_loss/len(test_loader)

  return avg_val_loss, correct

  # avg_val_losses.append(avg_val_loss)



In [ ]:
# Task 4: Define device, model, loss, optimizer:

model = NN4Layer(2000, 32, 1)

optimizer = Adam(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f'Using {device}')


In [ ]:
# Task 5: Start training for 20 epochs:

epochs = 20
avg_train_losses = []
avg_val_losses = []

for epoch in range(epochs):

  avg_train_loss = train_one_epoch(model, optimizer, loss_fn, train_loader, device)

  avg_val_loss, correct = validate(model, loss_fn, test_loader, device)

  avg_train_losses.append(avg_train_loss)
  avg_val_loss.append(avg_val_loss)

  if (epoch + 1) % 5 == 0:
    print(f'epoch: [{epoch + 1}]. train_loss: {avg_train_loss}. val_loss: {avg_val_loss}. labelled {correct/len(test_loader.dataset) * 100}% correctly')



In [ ]:
# Task 1: Write your code here:

plt.plot(avg_train_losses, label="Train Loss", c='red')
plt.plot(avg_val_losses, label="Validation Loss", c='purple')
plt.title("Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here:

